# Both Pick: models and group ranking

How do average, lower-score and mutual-threshold ranking change recommendations? This notebook loads saved models; it does not train them. Predictions estimate individual ratings, not enjoyment together.


In [1]:
import sys
import json
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
import altair as alt
from src import data as d, models as m, group as g, plots as pl

pd.set_option("display.width", 150, "display.max_columns", 30)
ratings, movies, genres = d.load_ratings(), d.load_movies(), d.load_movie_genres()
split = pd.read_parquet(d.PROCESSED_DIR / "split.parquet")
ctx = g.build_context(ratings, split, movies, genres)
metrics = json.loads((m.MODELS_DIR / "metrics.json").read_text())
models = {
    n: m.load_model(m.MODELS_DIR / f"{n}.npz")
    for n in ["baseline_global_mean", "baseline_bias", "matrix_factorization"]
}
OUT = Path.cwd().parent / "outputs"
(OUT / "figures").mkdir(parents=True, exist_ok=True)
(OUT / "tables").mkdir(exist_ok=True)


def save_chart(chart, name):
    chart.save(str(OUT / "figures" / f"{name}.html"))
    return chart


def save_table(df, name):
    df.to_csv(OUT / "tables" / f"{name}.csv")


## 1. Rating-prediction results

Ratings are split randomly within each user: about 70% training, 10% validation and 20% test. Training fits the models; validation selects hyperparameters; test measures held-out rating error. This is not a chronological evaluation.

The models are a global mean, regularized user/movie biases and matrix factorization. Missing ratings are not zero targets. Predictions are clipped to 0.5–5 stars. For a movie unseen in training, the item terms are omitted; the prediction uses the global mean and user bias.


In [2]:
rows = []
for part in ["val", "test"]:
    for name, r in metrics[part].items():
        rows.append(
            {
                "split": part,
                "model": name,
                "n_ratings": r["n_ratings"],
                "MAE": r["all"]["mae"],
                "RMSE": r["all"]["rmse"],
                "warm_coverage": r["warm_coverage"],
                "unseen_users": r["n_unseen_users"],
                "unseen_items": r["n_unseen_items"],
                "MAE_warm_only": r["warm_only"]["mae"],
                "RMSE_warm_only": r["warm_only"]["rmse"],
                "MAE_unseen_items": r["unseen_item_only"]["mae"],
            }
        )
results = pd.DataFrame(rows)
display(results[["split", "model", "n_ratings", "MAE", "RMSE", "unseen_items"]].round(4))
print("Split sizes:", metrics["split"]["rows"], "| scheme:", metrics["split"]["scheme"])
print("Selected on validation:", metrics["selected"])
t = results[results.split == "test"].melt(
    id_vars="model", value_vars=["MAE", "RMSE"], var_name="metric", value_name="error"
)
save_table(results, "model_results")
save_chart(
    alt.Chart(t)
    .mark_bar()
    .encode(
        x=alt.X(
            "model:N",
            title=None,
            sort=["baseline_global_mean", "baseline_bias", "matrix_factorization"],
            axis=alt.Axis(labelAngle=-20),
        ),
        y=alt.Y("error:Q", title="Test error (stars)"),
        color=alt.Color(
            "model:N", legend=None, scale=alt.Scale(range=["#999999", "#E69F00", "#0072B2"])
        ),
        column=alt.Column("metric:N", title=None),
        tooltip=["model", "metric", alt.Tooltip("error:Q", format=".4f")],
    )
    .properties(
        title=f"Held-out test error, n = {metrics['test']['matrix_factorization']['n_ratings']:,} ratings",
        width=200,
        height=240,
    ),
    "model_test_errors",
)


,split,model,n_ratings,MAE,RMSE,unseen_items
0,val,baseline_global_mean,77708,0.8336,1.0501,1042
1,val,baseline_bias,77708,0.6606,0.8648,1042
2,val,matrix_factorization,77708,0.6446,0.8468,1042
3,test,baseline_global_mean,155565,0.8355,1.0539,2093
4,test,baseline_bias,155565,0.6625,0.8703,2093
5,test,matrix_factorization,155565,0.6458,0.8508,2093


Split sizes: {'train': 544783, 'val': 77708, 'test': 155565} | scheme: random within each user (NOT chronological)
Selected on validation: {'bias': {'model': 'bias', 'lam_item': 5, 'lam_user': 2, 'val_rmse': 0.8647885004948194, 'val_mae': 0.6605567066882947}, 'mf': {'model': 'mf', 'n_factors': 50, 'reg': 0.02, 'val_rmse': 0.846793639641463, 'val_mae': 0.6445694200500369}}


alt.Chart(...)

## 2. Compare ranking strategies

Use the same coverage-selected pair as notebook 01. All strategies start with the same movies: at least 10 training ratings per movie, excluding either user’s training history.

- Average: `(A + B) / 2`.
- Lower score: `min(A, B)`.
- Mutual threshold: require both predictions to reach 3.5, then rank by the average.

Ties use average, lower score, then movie ID. The summary compares the top 10; the lists below show the first five. This example pair differs from the app’s default pair.


In [3]:
search = d.find_first_eligible_pair(ratings, 100, 200, 20)
ua, ub, n_shared = search["pair"]
mf = models["matrix_factorization"]
for u in (ua, ub):
    print(g.profile_summary(ctx, u)["n_train"], "training ratings for user", u)
cands = g.build_candidates(mf, ctx, ua, ub, exclude="training", min_item_train=10)
cands_all = g.build_candidates(mf, ctx, ua, ub, exclude="all_recorded", min_item_train=10)
print(
    f"Pair {ua} & {ub} ({n_shared} co-rated movies overall). Candidates (exclude training histories): {len(cands):,}; demo mode (exclude all recorded): {len(cands_all):,}"
)
MIN_A = MIN_B = 3.5
summary, overlap, tops = g.compare_strategies(cands, k=10, min_a=MIN_A, min_b=MIN_B)
display(summary.round(3))
print("Top-10 overlap (number of shared movies):")
display(overlap)
save_table(summary, "strategy_summary")
save_table(overlap, "strategy_top10_overlap")


292 training ratings for user 59
105 training ratings for user 83
Pair 59 & 83 (56 co-rated movies overall). Candidates (exclude training histories): 5,594; demo mode (exclude all recorded): 5,464


,n_selected,mean_avg_pred,mean_lower_pred,mean_abs_gap
average,10,4.317,3.885,0.863
lower,10,4.229,3.952,0.554
threshold,10,4.317,3.885,0.863


Top-10 overlap (number of shared movies):


,average,lower,threshold
average,10,4,10
lower,4,10,4
threshold,10,4,10


In [4]:
for strategy in g.STRATEGIES:
    print(g.STRATEGY_LABELS[strategy])
    display(g.display_table(tops[strategy].head(5)).round(2))


Average


,rank,movieId,title,genres,pred_a,pred_b,avg_pred,min_pred,score,item_train_n
0,1,170705,Band of Brothers (2001),"Action, Drama, War",3.98,4.79,4.39,3.98,4.39,47
1,2,26150,Andrei Rublev (Andrey Rublyov) (1969),"Drama, War",3.87,4.83,4.35,3.87,4.35,26
2,3,159817,Planet Earth (2006),Documentary,4.02,4.66,4.34,4.02,4.34,56
3,4,171495,Cosmos,(no genres listed),3.87,4.78,4.32,3.87,4.32,16
4,5,6643,Tokyo Story (Tôkyô monogatari) (1953),Drama,3.98,4.66,4.32,3.98,4.32,26


Lower score (least misery)


,rank,movieId,title,genres,pred_a,pred_b,avg_pred,min_pred,score,item_train_n
0,1,159817,Planet Earth (2006),Documentary,4.02,4.66,4.34,4.02,4.02,56
1,2,170705,Band of Brothers (2001),"Action, Drama, War",3.98,4.79,4.39,3.98,3.98,47
2,3,6643,Tokyo Story (Tôkyô monogatari) (1953),Drama,3.98,4.66,4.32,3.98,3.98,26
3,4,7748,Pierrot le fou (1965),"Crime, Drama",3.96,4.38,4.17,3.96,3.96,19
4,5,4914,Breathless (À bout de souffle) (1960),"Crime, Drama, Romance",3.95,4.39,4.17,3.95,3.95,52


Mutual threshold


,rank,movieId,title,genres,pred_a,pred_b,avg_pred,min_pred,score,item_train_n
0,1,170705,Band of Brothers (2001),"Action, Drama, War",3.98,4.79,4.39,3.98,4.39,47
1,2,26150,Andrei Rublev (Andrey Rublyov) (1969),"Drama, War",3.87,4.83,4.35,3.87,4.35,26
2,3,159817,Planet Earth (2006),Documentary,4.02,4.66,4.34,4.02,4.34,56
3,4,171495,Cosmos,(no genres listed),3.87,4.78,4.32,3.87,4.32,16
4,5,6643,Tokyo Story (Tôkyô monogatari) (1953),Drama,3.98,4.66,4.32,3.98,4.32,26


In the additive bias model, the difference between users’ predictions is constant before clipping. Average and lower-score ranking can therefore have the same order. Matrix factorization adds user–movie interactions, so the two strategies can differ.


In [5]:
bias_c = g.build_candidates(
    models["baseline_bias"], ctx, ua, ub, exclude="training", min_item_train=10
)
_, ov_b, _ = g.compare_strategies(bias_c, k=10, min_a=MIN_A, min_b=MIN_B)
print("Bias baseline, top-10 overlap:")
display(ov_b)


Bias baseline, top-10 overlap:


,average,lower,threshold
average,10,10,10
lower,10,10,10
threshold,10,10,10


In [6]:
save_chart(
    pl.pair_scatter(cands, tops["average"], ua, ub, "Average strategy: top 10"),
    "pair_predictions",
)


alt.LayerChart(...)

## 3. Check strategies against held-out ratings

Select up to 500 pairs from the 1,000 lowest-ID users with at least 50 training ratings. Each pair must share at least five test movies; selection uses coverage, not rating values.

Rank those shared test movies using predictions, then compare the selected top three against the users’ recorded test ratings. The reference is the mean over all shared test candidates. This is a different pool from the app’s discovery recommendations.

Thresholds can leave fewer selections or exclude a pair entirely, so compare threshold results on matching pairs. These co-rated movies are self-selected and may not represent other movies or typical user pairs.


In [7]:
pairs, info = g.select_pairs(
    ctx, min_train=50, pool_size=1000, min_test_shared=5, max_pairs=500, seed=830
)
print(info)
ev = g.evaluate_pairs(
    {"bias": models["baseline_bias"], "mf": models["matrix_factorization"]},
    ctx,
    pairs,
    k=3,
    min_a=3.5,
    min_b=3.5,
)
print(
    f"Pairs evaluated: {len(ev)}; candidates per pair: min {ev.n_candidates.min()}, median {ev.n_candidates.median():.0f}, max {ev.n_candidates.max()}"
)
out = []
for model in ["bias", "mf"]:
    for s in g.STRATEGIES:
        n = ev[f"{model}|{s}|n"]
        out.append(
            {
                "model": model,
                "strategy": s,
                "pairs_with_selection": int((n > 0).sum()),
                "mean_obs_avg_top3": ev[f"{model}|{s}|obs_avg"].mean(),
                "mean_obs_lower_top3": ev[f"{model}|{s}|obs_min"].mean(),
                "pairs_top3_avg_above_pool": float(
                    (ev[f"{model}|{s}|obs_avg"] > ev["pool_mean_obs_avg"]).mean()
                ),
            }
        )
res = pd.DataFrame(out)
ref = pd.DataFrame(
    [
        {
            "model": "all candidates",
            "strategy": "reference",
            "pairs_with_selection": len(ev),
            "mean_obs_avg_top3": ev.pool_mean_obs_avg.mean(),
            "mean_obs_lower_top3": ev.pool_mean_obs_min.mean(),
            "pairs_top3_avg_above_pool": np.nan,
        }
    ]
)
display(
    pd.concat([res, ref])[
        ["model", "strategy", "pairs_with_selection", "mean_obs_avg_top3", "mean_obs_lower_top3"]
    ].round(3)
)
mask = ev["mf|threshold|n"] > 0
print(
    f"Like-for-like: on the {int(mask.sum())} pairs where the threshold strategy selected something, MF average top-3 observed avg = {ev.loc[mask, 'mf|average|obs_avg'].mean():.3f}, "
    f"threshold = {ev.loc[mask, 'mf|threshold|obs_avg'].mean():.3f}; pool mean = {ev.loc[mask, 'pool_mean_obs_avg'].mean():.3f}. "
    "(The threshold rows above average over a different subset of pairs, so they are not directly comparable to the other rows.)"
)
save_table(pd.concat([res, ref]), "pair_evaluation")
save_table(ev, "pair_evaluation_per_pair")


{'pool_users': 1000, 'pairs_possible': 499500, 'pairs_eligible': 29368, 'pairs_used': 500}


Pairs evaluated: 500; candidates per pair: min 5, median 6, max 29


,model,strategy,pairs_with_selection,mean_obs_avg_top3,mean_obs_lower_top3
0,bias,average,500,3.955,3.488
1,bias,lower,500,3.955,3.488
2,bias,threshold,403,4.105,3.688
3,mf,average,500,3.980,3.532
4,mf,lower,500,3.971,3.529
5,mf,threshold,436,4.101,3.698
0,all candidates,reference,500,3.660,3.169


Like-for-like: on the 436 pairs where the threshold strategy selected something, MF average top-3 observed avg = 4.043, threshold = 4.101; pool mean = 3.716. (The threshold rows above average over a different subset of pairs, so they are not directly comparable to the other rows.)


## Interpretation to finish

Explain the rating-error results and the tradeoffs between the ranking strategies using the computed evidence. State what the simulated-pair evaluation can and cannot establish.

New-user support, genre-based fallback and chronological evaluation are not implemented. The genre filter is not a cold-start method.
